# PaLiGemma Rwanda Crop Disease Diagnosis - Test Notebook

This notebook tests the PaLiGemma model fine-tuned with LoRA for recommending treatments to Rwandan farmers based on crop leaf images.

**Your trained adapter:** `paligemma-rwanda-lora` (trained with q_proj LoRA, rank=16)

**Requirements:**
- Google Colab with GPU runtime (T4 or better)
- Hugging Face account for accessing PaLiGemma base model

## 1. Setup Environment

In [ ]:
# Install required packages
!pip install -q torch torchvision
!pip install -q transformers>=4.42.0 accelerate>=0.26.0
!pip install -q peft>=0.13.0
!pip install -q huggingface-hub pillow

In [ ]:
# Login to Hugging Face (required for PaLiGemma access)
from huggingface_hub import login

# You can get your token from: https://huggingface.co/settings/tokens
# Make sure you've accepted the PaLiGemma license at: https://huggingface.co/google/paligemma2-3b-pt-448
login()

In [ ]:
import torch
import os

# Check GPU availability
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Go to Runtime > Change runtime type > GPU")

## 2. Upload Your Trained LoRA Adapter

Upload the `paligemma-rwanda-lora.zip` file you downloaded from your training session.

In [ ]:
from google.colab import files
import zipfile
import os

# Upload the zip file
print("Upload your paligemma-rwanda-lora.zip file:")
uploaded = files.upload()

# Extract
if 'paligemma-rwanda-lora.zip' in uploaded:
    with zipfile.ZipFile('paligemma-rwanda-lora.zip', 'r') as zip_ref:
        zip_ref.extractall('.')
    print("\nExtracted adapter files:")
    !ls -la artifacts/paligemma-rwanda-lora/
else:
    print("Please upload paligemma-rwanda-lora.zip")

In [ ]:
# Set the adapter path
ADAPTER_PATH = "artifacts/paligemma-rwanda-lora"

# Verify the adapter files exist
required_files = ["adapter_config.json", "adapter_model.safetensors"]
for f in required_files:
    path = os.path.join(ADAPTER_PATH, f)
    if os.path.exists(path):
        size = os.path.getsize(path) / 1024 / 1024
        print(f"Found: {f} ({size:.2f} MB)")
    else:
        print(f"Missing: {f}")

## 3. Load PaLiGemma Model with Your LoRA Adapter

In [ ]:
from transformers import AutoProcessor, PaliGemmaForConditionalGeneration
from peft import PeftModel
from PIL import Image
import requests
from io import BytesIO

# Model configuration
BASE_MODEL = "google/paligemma2-3b-pt-448"

# Set device and dtype
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

print(f"Using device: {device}")
print(f"Using dtype: {dtype}")

In [ ]:
# Load processor from your adapter (includes tokenizer)
print("Loading processor from adapter...")
processor = AutoProcessor.from_pretrained(ADAPTER_PATH, use_fast=False)
print("Processor loaded!")

In [ ]:
# Load base model
print("Loading base PaLiGemma model (this may take a few minutes)...")
base_model = PaliGemmaForConditionalGeneration.from_pretrained(
    BASE_MODEL,
    torch_dtype=dtype,
    device_map="auto",
    low_cpu_mem_usage=True,
)
print("Base model loaded!")

In [ ]:
# Load your trained LoRA adapter
print("Loading LoRA adapter...")
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH, is_trainable=False)
model.eval()
print("LoRA adapter loaded!")
print("Model ready for inference!")

## 4. Define Inference Functions

In [ ]:
import re

def build_prompt(crop_type: str = "crop") -> str:
    """Build the prompt for disease diagnosis."""
    return (
        f"<image> A farmer uploaded this {crop_type} leaf image from Rwanda. "
        "Identify the likely disease and provide practical recommendation. "
        "Format exactly as: Disease: <name>. Advice: <recommendation>."
    )

def parse_response(text: str) -> dict:
    """Parse the model's response to extract disease and advice."""
    normalized = re.sub(r"\s+", " ", text.strip())
    
    disease_match = re.search(r"disease\s*:\s*(.+?)(?=\.\s*advice|\sadvice|$)", normalized, re.I)
    advice_match = re.search(r"advice\s*:\s*(.+?)(?=\.\s*source|\ssource|$)", normalized, re.I)
    
    return {
        "diagnosis": disease_match.group(1).strip(" .") if disease_match else normalized.split(".")[0],
        "recommendation": advice_match.group(1).strip() if advice_match else normalized,
        "raw_text": normalized
    }

def diagnose_crop(image: Image.Image, crop_type: str = "crop") -> dict:
    """Run inference on a crop leaf image."""
    prompt = build_prompt(crop_type)
    
    # Prepare inputs
    inputs = processor(images=image, text=prompt, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    # Generate response
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=180,
            do_sample=False,
        )
    
    # Decode only the generated part (skip prompt)
    prompt_len = inputs["input_ids"].shape[1]
    generated_ids = output_ids[0][prompt_len:]
    generated_text = processor.tokenizer.decode(generated_ids, skip_special_tokens=True)
    
    return parse_response(generated_text)

print("Inference functions defined!")

## 5. Test with Your Images

In [ ]:
# Upload a crop leaf image for diagnosis
print("Upload a crop leaf image:")
uploaded = files.upload()

if uploaded:
    filename = list(uploaded.keys())[0]
    test_image = Image.open(BytesIO(uploaded[filename])).convert("RGB")
    print(f"\nImage loaded: {filename} ({test_image.size})")
    
    # Display the image
    display(test_image.resize((300, 300)))

In [ ]:
# Run diagnosis
print("Running diagnosis with your trained model...")
print("=" * 50)

# Specify crop type: "beans", "maize", or "crop" (auto)
result = diagnose_crop(test_image, crop_type="beans")

print(f"\n🌱 DIAGNOSIS: {result['diagnosis']}")
print(f"\n💡 RECOMMENDATION:")
print(result['recommendation'])
print(f"\n📝 Raw model output:")
print(result['raw_text'])

In [ ]:
# Test with maize
print("Testing with maize crop type...")
print("=" * 50)

result_maize = diagnose_crop(test_image, crop_type="maize")

print(f"\n🌽 DIAGNOSIS: {result_maize['diagnosis']}")
print(f"\n💡 RECOMMENDATION:")
print(result_maize['recommendation'])

## 6. Batch Testing - Multiple Images

In [ ]:
# Upload multiple images for batch testing
print("Upload multiple crop leaf images:")
uploaded_batch = files.upload()

results = []
for filename, content in uploaded_batch.items():
    print(f"\nProcessing: {filename}")
    print("-" * 40)
    
    image = Image.open(BytesIO(content)).convert("RGB")
    result = diagnose_crop(image, crop_type="crop")  # Auto-detect
    result["filename"] = filename
    results.append(result)
    
    print(f"  Diagnosis: {result['diagnosis']}")
    print(f"  Recommendation: {result['recommendation'][:100]}...")

print(f"\nProcessed {len(results)} images")

## 7. Export Results

In [ ]:
import json
import pandas as pd

# Convert results to DataFrame
if results:
    df = pd.DataFrame(results)
    display(df)
    
    # Save to CSV
    df.to_csv("diagnosis_results.csv", index=False)
    print("\nResults saved to diagnosis_results.csv")
    
    # Download
    files.download("diagnosis_results.csv")

## 8. Cleanup (Optional)

In [ ]:
# Free GPU memory
import gc

del model
del base_model
del processor
gc.collect()
torch.cuda.empty_cache()

print("GPU memory cleared!")

---
## Model Information

### Training Details:
- **Base model**: google/paligemma2-3b-pt-448
- **LoRA rank**: 16
- **LoRA alpha**: 32
- **Target modules**: q_proj
- **Trainable parameters**: ~5.4M (0.18% of total)

### Supported Crops:
- **Beans**: angular_leaf_spot, bean_rust, healthy
- **Maize**: common_rust, gray_leaf_spot, northern_leaf_blight, healthy

### Tips for Better Results:
1. Take clear photos in good lighting
2. Focus on the affected leaf area
3. Avoid blurry or dark images
4. Include both healthy and diseased parts if visible